# Stitch viewer: tiles, design, contours and design errors, before and after stitching

Runs the pipeline on the chosen tiles and opens one interactive window (pyqtgraph, OpenGL):

1. **Per tile** (`process_tile`): load the SEM image, detect contacts (Otsu), refine the contours
   to the maximum gradient, read the design contacts from the tile's `.oas` in the tone (normal /
   reversed) that matches the SEM contacts. Tiles where neither tone matches are flagged.
2. **Stitching**, translation per tile, failed overlaps and unstitched tiles listed:
   the SEM tiles (`stitch_tiles`) and, the same way, the per-tile design files (`stitch_design`),
   which can be offset against each other.
3. **Design errors** (`design_errors`) for each placement (`placement_corrections`): nominal,
   stitched (corrections average 0), stitched with the first tile fixed. Raw error SEM − design,
   no affine removed yet. Tiles that could not be measured are listed with the reason.
4. **Viewer** (`StitchViewer`): left, SEM images, design contours/centres, Otsu and refined
   contours/centres, overlap boxes and overlap outliers; right, the error map of the selected
   placement, sharing the zoom. Layers and single tiles can be hidden on the right.

The window blocks this notebook until it is closed. Positions in µm, errors in nm.

In [ ]:
DATA_DIR = r"A:\Stitch_monitor\In_cell_without_PTM_20260914\260910_1143_Norm"
TILES = None              # None = all tiles, or a list of indices in read_tile_index order, e.g. list(range(10))
LAYER = (100, 0)          # (layer, datatype) of the contacts in the .oas
SEARCH_PX = 5.0           # refinement searches this many pixels in/out along the contour normal
DESIGN_MATCH_NM = 40.0    # pair design and SEM contacts closer than this (tone check and design errors)
MIN_MATCH_FRACTION = 0.5  # a design tone is accepted if at least this fraction of SEM contacts match it
STITCH_REFINED = True     # stitch with the refined centres (False: Otsu centres)
MAX_MATCH_NM = 25.0       # SEM overlaps: pair contacts closer than this, the largest B - A stage offset (< pitch / 2)
DESIGN_MAX_MATCH_NM = 25.0  # design overlaps: the largest offset between neighbouring .oas files (< pitch / 2)
OUTLIER_FACTOR = 3.0      # overlap outlier if residual > this x the median residual
MIN_MATCHED = 5           # pairs with fewer matched contacts are not used (reported below)
ARROW_SCALE = 10          # error-map lines are drawn this many times longer than the error
USE_OPENGL = True         # set False if the viewer window stays black

In [ ]:
import time

import numpy as np
import pandas as pd
import pyqtgraph as pg

from affine_ransac.io.design import load_layout, read_polygons
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.pipeline import process_tile, stitch_design, stitch_tiles
from affine_ransac.registration import design_errors, error_summary
from affine_ransac.stitching import placement_corrections
from affine_ransac.view_stitch import StitchViewer

## 1. Load the tiles and detect their contacts

In [ ]:
all_tiles = read_tile_index(DATA_DIR)
records = all_tiles if TILES is None else [all_tiles[i] for i in TILES]
tile_ids = [r.tile_id for r in records]

start = time.perf_counter()
tiles = [
    process_tile(r.image_path, r.oas_path, (r.center_x_nm, r.center_y_nm), (r.fov_x_nm, r.fov_y_nm),
                 LAYER, SEARCH_PX, DESIGN_MATCH_NM, MIN_MATCH_FRACTION)
    for r in records
]
print(f"{len(tiles)} of {len(all_tiles)} tiles in {time.perf_counter() - start:.1f} s")

per_tile = pd.DataFrame({
    "tile": tile_ids,
    "design": [len(t.design.centers) for t in tiles],
    "otsu": [len(t.otsu.centers) for t in tiles],
    "refined": [len(t.refined.centers) for t in tiles],
    "tone": ["reversed" if t.design.reversed else "normal" for t in tiles],
    "match_fraction": [t.design.match_fraction for t in tiles],
    "design_ok": [t.design.ok for t in tiles],
})
print(f"Design tone: {(per_tile.tone == 'normal').sum()} normal, {(per_tile.tone == 'reversed').sum()} reversed")
flagged = per_tile[~per_tile.design_ok]
if len(flagged):
    print(f"\n!! {len(flagged)} tile(s) FLAGGED: neither design tone matches the SEM (or no .oas):")
    display(flagged.round(2))
per_tile.describe().round(2)

## 2. Stitching: SEM tiles and design files

In [ ]:
centers = np.array([t.center_nm for t in tiles])
fovs = np.array([t.fov_nm for t in tiles])
stitch = stitch_tiles([t.sem_points_nm(STITCH_REFINED) for t in tiles], centers, fovs,
                      MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)
design_stitch = stitch_design(tiles, DESIGN_MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)

for name, result in (("SEM", stitch), ("Design", design_stitch)):
    failed = [r for r in result.rejected if r.failed]
    print(f"{name}: {len(result.pairs)} overlapping pairs used, {len(failed)} not matched, "
          f"{len(result.rejected) - len(failed)} skipped (fewer than {MIN_MATCHED} contacts in the overlap); "
          f"corrections |x| max {np.nanmax(np.abs(result.corrections[:, 0])):.2f} nm, "
          f"|y| max {np.nanmax(np.abs(result.corrections[:, 1])):.2f} nm")
    if failed or len(result.unplaced):
        print(f"!! {name.upper()} STITCHING INCOMPLETE. Tiles not stitched: {[tile_ids[k] for k in result.unplaced]}")
        display(pd.DataFrame([{"tile_a": tile_ids[r.i], "tile_b": tile_ids[r.j], "contacts_in_overlap": r.in_box,
                               "matched": r.matched} for r in failed]))

pd.DataFrame({
    "tile": tile_ids,
    "sem_correction_x_nm": stitch.corrections[:, 0], "sem_correction_y_nm": stitch.corrections[:, 1],
    "design_correction_x_nm": design_stitch.corrections[:, 0], "design_correction_y_nm": design_stitch.corrections[:, 1],
}).round(2)

## 3. Design errors per placement (raw: SEM − design, no affine removed)

A contact seen by two tiles in an overlap has one entry per tile. The common offset depends on
the placement, so look at the pattern: a uniform shift, a rotation (arrows circling) or a scale
(arrows pointing outward/inward).

In [ ]:
sem_points = [t.sem_points_nm(STITCH_REFINED) for t in tiles]
design_points = [t.design_points_nm() for t in tiles]
design_ok = [t.design.ok for t in tiles]
errors = {
    placement: design_errors(sem_points, design_points, sem_corrections, design_corrections, design_ok, DESIGN_MATCH_NM)
    for placement, (sem_corrections, design_corrections) in placement_corrections(stitch.corrections,
                                                                                  design_stitch.corrections).items()
}

for placement, e in errors.items():
    if e.skipped:
        print(f"!! {placement}: {len(e.skipped)} tile(s) NOT MEASURED (no error computed):")
        display(pd.DataFrame([{"tile": tile_ids[k], "reason": reason} for k, reason in e.skipped.items()]))
pd.DataFrame({placement: error_summary(e.error_nm) for placement, e in errors.items()}).round(2)

## 4. Viewer (closes with the window)

Left: SEM and design. Right: the error map of the selected placement (dot = design position,
line = error × `ARROW_SCALE`, adjustable in the panel). Both plots share zoom and pan.

In [ ]:
design_polygons = [read_polygons(load_layout(r.oas_path), *LAYER) if r.oas_path else [] for r in records]

app = pg.mkQApp("Stitch viewer")
viewer = StitchViewer(tiles, tile_ids, design_polygons, stitch, design_stitch, refined=STITCH_REFINED,
                      use_opengl=USE_OPENGL, errors=errors, arrow_scale=ARROW_SCALE)
viewer.show()
app.exec()